# Regularization

**Titanic** passengers - let's deepen our understanding of the factors affecting survival chances
- We'll use logistic classifiers, which are easy to interpret
- We did this before with statsmodels in the "Decision Science - Logistic Regression" lecture
- We used `p-values` and statistical assumptions to identify which features were irrelevant / couldn't be generalized
- This time, we'll use `regularization` to identify relevant/irrelevant features based on underfitting/overfitting criteria
- **Our goal is to compare `L1` and `L2` penalties**

## 1. We load and preprocess the data for you

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ML_titanic_dataset_encoded.csv")

# the dataset is already one-hot-encoded
data.head()

,survived,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,1,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,0,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [3]:
# We build X and y

y = data["survived"]
X = data.drop(columns=["survived"])
X.head()

,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [4]:
# We MinMaxScale our features for you
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler().fit(X)
X_scaled = scaler.transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X.shape

(714, 12)

## 2. Logistic Regression without Regularization

❓ Train a simple **unregularized** Logistic Regression and rank the features by importance in descending order (i.e., look at the coefficients after training)
- Note: `LogisticRegression` is penalized by default
  - See the [penalty parameter](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) to learn how to remove the penalty)
- Increase `max_iter` to a larger number until the model converges
- Use `tol=1e-9` to set the solver's stopping criterion: the solver will stop when the largest component of the gradient is smaller than this. If you set it to higher values, you'll see the coefficients fluctuate a lot with `tol`.

<details>
    <summary>Hint</summary>
    <img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/05-ML/05-Model-Tuning/model_selection.png" alt="penalizing a regression" width="500">
</details>

In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(penalty=None, max_iter=1000, tol=1e-9)
model.fit(X_scaled, y)
pd.Series(model.coef_.tolist()[0], index=X_scaled.columns).sort_values(ascending=False)

/Users/yaren/.pyenv/versions/workintech/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


pclass                      5.451110
class_First                 3.812356
sex_female                  2.671880
fare                        1.360201
who_child                   1.336357
parch                      -0.894276
age                        -2.196128
sibsp                      -2.476886
class_Third                -3.908752
embark_town_Cherbourg     -21.385507
embark_town_Southampton   -21.686876
embark_town_Queenstown    -22.082281
dtype: float64

❓ How would you interpret the value of the `sex_female` coefficient in plain English?

<details>
    <summary>Answer</summary>

> "All else being equal (age, ticket class, etc...),
being female increases your log-odds of survival by 2.67 (your coefficient value)"
    
> "While controlling for all other explanatory factors present in this dataset,
being female multiplies your survival odds by exp(2.67) = 14"

</details>

❓ According to your model, which feature has the most impact on survival chances?  
Fill in the `top_1_feature` list below with the name of that feature

In [7]:
top_1_feature = [""]

In [10]:
top_1_feature = ["embark_town_Queenstown"]

In [11]:
from nbresult import ChallengeResult
result = ChallengeResult('unregularized', top_1_feature=top_1_feature)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/yaren/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/yaren/code/ds_projects/lasso-regularization/tests
plugins: anyio-4.12.1, dash-4.0.0, typeguard-4.4.2
collecting ... collected 1 item

test_unregularized.py::TestUnregularized::test_top_1 PASSED              [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/unregularized.pickle

git commit -m 'Completed unregularized step'

git push origin master



## 3. Logistic Regression with L2 Penalty

Let's use a **Logistic model** penalized with a log-loss **L2** term to find the **most important features** without overfitting.  
This is the "classification" counterpart of the "Ridge" regressor

❓ Create a **strongly regularized** `LogisticRegression` and rank its features by importance (look at the coefficients)
- By "strongly regularized" we mean "more than Sklearn's default regularization factor"
- Sklearn's default values are very useful orders of magnitude to keep in mind for "scaled features"

In [12]:
model = LogisticRegression(penalty="l2", C=0.1, max_iter=1000, tol=1e-9)
model.fit(X_scaled, y)
pd.Series(model.coef_.tolist()[0], index=X_scaled.columns).sort_values(ascending=False)

/Users/yaren/.pyenv/versions/workintech/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


sex_female                 1.808561
who_child                  0.602854
class_First                0.441520
embark_town_Cherbourg      0.252948
fare                       0.136793
parch                     -0.053898
embark_town_Queenstown    -0.132384
embark_town_Southampton   -0.154404
sibsp                     -0.340862
age                       -0.477739
pclass                    -0.539221
class_Third               -0.636922
dtype: float64

❓ According to your model, what are the top 2 features affecting survival chances?  
Fill in the `top_2_features` list below with the names of those features

In [ ]:
top_2_features = ["", ""]

In [14]:
top_2_features = ["sex_female", "class_Third"]

#### 🧪 Test your code below

In [15]:
from nbresult import ChallengeResult
result = ChallengeResult('ridge', top_2=top_2_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/yaren/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/yaren/code/ds_projects/lasso-regularization/tests
plugins: anyio-4.12.1, dash-4.0.0, typeguard-4.4.2
collecting ... collected 1 item

test_ridge.py::TestRidge::test_top2 PASSED                               [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/ridge.pickle

git commit -m 'Completed ridge step'

git push origin master



## 4. Logistic Regression with L1 Penalty

This time, we'll use a logistic model penalized with a log-loss **L1** term to **filter out less important features**.  
This is the "classification" counterpart of the **Lasso** regressor

❓ Create a **strongly regularized** `LogisticRegression` and rank its features by importance

In [ ]:
# YOUR CODE HERE

❓ According to your L1 model, which features have absolutely zero impact on survival chances?  
Fill in the `zero_impact_features` list below with the names of those features; you may need to add more elements to the list.

- Did you notice that some of them were "very important" according to the unregularized model? 
- From now on, we'll always regularize our linear models!

In [ ]:
zero_impact_features = ["", "", "", ""]

#### 🧪 Test your code below

In [ ]:
from nbresult import ChallengeResult
result = ChallengeResult('lasso', zero_impact_features = zero_impact_features)
result.write()
print(result.check())

# 5. Taking a step back

🤯 **Why were some of these coefficients so high in the first place?**

Consider the three features that were zeroed out by regularization:
- `embark_town_Cherbourg`
- `embark_town_Southampton`
- `embark_town_Queenstown`

The three embarkation towns are of course related: if you didn't board from two of them, you must have boarded from the third. So we know that: 

$$embark\_town\_Cherbourg + embark\_town\_Southampton + embark\_town\_Queenstown = 1$$

These three features are **perfectly multicollinear**!

**When using unregularized models, this often leads to numerical instability**, which is exactly what we saw here. It also means we **can't really trust the coefficients** we obtained in that case.

❗️ These three multicollinear features come from the one-hot encoding of the `embark_town` categorical feature.

Regularization helped us overcome this problem: it prevented the coefficients for the three towns from becoming very large. **That's why we'll almost always use regularization.**

🔍 **Remember the `tol` parameter we set at the start?**

An extra bonus of regularization is that tuning `tol` becomes less important: you can set it to any value between `1e-2` and `1e-9` and the coefficients barely change! 💪

**🏁 Congratulations! Don't forget to commit and push your notebook**